# NB_08_trial_builder - Trial Builder

**Purpose.** Author and persist TrialDefinition objects (closed-loop routes) without running any simulation. Includes an interactive waypoint picker that snaps to the nearest map waypoint.

**Inputs.** a CARLA server (only for the live waypoint picker), an existing scenario name (optional, to clone).

**Outputs.** MIREIA/trials/<name>/trial.json.

**How to run.** Run sections 0-5 to author a single trial, or section 7 to auto-generate trials from a scenario range. The route picker is launched as a subprocess.

**Position in the workflow.** Authoring step before NB_09 (single live run) and NB_10-13 (batch runners).


## 0 — CARLA Launch

In [ ]:
import subprocess
subprocess.Popen("CarlaUE4.exe", shell=True)

## 1 — Imports

In [ ]:
import json
import re
import sys
from pathlib import Path

import carla

from MIREIA.config import Config
from MIREIA.simulation.trials import TrialDefinition


## 1 — CARLA Connection
Start CARLA before running this cell. Change `MAP_NAME` to load a different map.

In [ ]:
client = carla.Client(Config.CARLA_HOST, Config.CARLA_PORT)
client.set_timeout(15.0)

In [ ]:
MAP_NAME = "Town10HD"  # change to any available map



current_map = client.get_world().get_map().name.split("/")[-1]
if current_map != MAP_NAME:
    print(f"Loading {MAP_NAME} (current: {current_map})...")
    client.load_world(MAP_NAME)

world = client.get_world()
print(f"Connected. Map: {world.get_map().name.split('/')[-1]}")
print("Available maps:", [m.split("/")[-1] for m in client.get_available_maps()])

## 2 — Existing Trials
Lists already-saved trial definitions.

In [ ]:
trials_root = Path(Config.PATH_TO_TRIALS)
existing = sorted(p for p in trials_root.glob("*/trial.json") if p.is_file())

if not existing:
    print("No saved trials found.")
else:
    print(f"Found {len(existing)} trial(s):\n")
    for p in existing:
        data = json.loads(p.read_text(encoding="utf-8"))
        # Backwards compat: old trials only stored route_start/route_end.
        if "route_waypoints" in data:
            wps = [tuple(w) for w in data["route_waypoints"]]
        else:
            wps = [tuple(data.get("route_start", [])), tuple(data.get("route_end", []))]
        wps_str = " -> ".join(f"({w[0]:.1f}, {w[1]:.1f})" for w in wps)
        print(
            f"  {data['name']!r:40s}  "
            f"map={data.get('map_name','?')}  "
            f"weather={data.get('weather','?')}  "
            f"nv={data.get('n_vehicles','?')}  "
            f"route ({len(wps)} pts): {wps_str}"
        )

## 3 — Pick Route Points
Launches a separate window showing the map waypoints.

- **Left-click** to snap each click to the nearest waypoint. First click = start (green), second = end (red).
- **Right-click** to undo the last selected waypoint.
- Close the window or press **Enter** once both points are picked.

The picked coordinates are parsed and stored in `route_start` and `route_end` automatically.

In [ ]:
import os

routes_script = Path("MIREIA/simulation/routes.py")

# Override the matplotlib backend so the subprocess opens a real interactive
# window instead of inheriting Jupyter's inline backend (which closes instantly).
env = os.environ.copy()
env["MPLBACKEND"] = "TkAgg"

result = subprocess.run(
    [sys.executable, str(routes_script), "--host", Config.CARLA_HOST, "--port", str(Config.CARLA_PORT)],
    capture_output=True,
    text=True,
    env=env,
)

output = result.stdout + result.stderr
print(output.strip())

def _parse_carla_location(text: str, label: str) -> tuple[float, float, float] | None:
    pattern = rf"{label}\s*=\s*carla\.Location\(x=([\-\d.]+),\s*y=([\-\d.]+),\s*z=([\-\d.]+)\)"
    m = re.search(pattern, text)
    if m:
        return float(m.group(1)), float(m.group(2)), float(m.group(3))
    return None

route_start = _parse_carla_location(output, "START_CARLA")
route_end   = _parse_carla_location(output, "END_CARLA")

if route_start and route_end:
    print(f"\nParsed start: {route_start}")
    print(f"Parsed end:   {route_end}")
else:
    print("\n[warn] Could not parse coordinates - copy them from the output above and set route_start/route_end manually.")
    route_start = (0.0, 0.0, 0.0)
    route_end   = (0.0, 0.0, 0.0)

### Manual Override
If the picker didn't run or you want to set coordinates by hand, edit and run this cell.

In [ ]:
# route_start = (7.409, -64.400, 0.0)
# route_end   = (44.003, 50.549, 0.0)
print(f"route_start = {route_start}")
print(f"route_end   = {route_end}")

## 4 — Trial Parameters
Edit these to define the trial. `name` must be unique — it becomes the folder name under `PATH_TO_TRIALS`.

Weather presets: `ClearNoon`, `CloudyNoon`, `WetNoon`, `HardRainNoon`, `CloudySunset`, `SoftRainSunset`, `ClearNight`, `CloudyNight`, `HardRainNight`.

In [ ]:
TRIAL_NAME   = "my_trial_town10HD_clear"  # unique folder name
DESCRIPTION  = "Town10HD, clear noon, moderate traffic."
WEATHER      = "ClearNoon"
N_VEHICLES   = 30
N_PEDESTRIANS = 20
SEED         = 42

# Advanced — leave as defaults unless you need specific traffic behaviour
PCT_RUNNING   = 0.0   # fraction of pedestrians that run
PCT_CROSSING  = 0.0   # fraction of pedestrians that ignore traffic lights
SAFE_VEHICLES = True  # avoid collisions during vehicle spawning

## 5 — Create and Save Trial

In [ ]:
trial = TrialDefinition(
    name=TRIAL_NAME,
    route_start=route_start,
    route_end=route_end,
    description=DESCRIPTION,
    map_name=MAP_NAME,
    weather=WEATHER,
    n_vehicles=N_VEHICLES,
    n_pedestrians=N_PEDESTRIANS,
    pct_running=PCT_RUNNING,
    pct_crossing=PCT_CROSSING,
    safe_vehicles=SAFE_VEHICLES,
    seed=SEED,
    sync_mode=True,
    fixed_delta=Config.SIM_FIXED_DELTA_SECONDS,
)

trial.save()
print(f"Saved: {trial.json_path}")
print(json.dumps(trial.to_dict(), indent=2))

## 6 — Review / Load an Existing Trial
Load any saved trial by name to inspect or copy its parameters.

In [ ]:
load_name = TRIAL_NAME  # or replace with any other trial name

loaded = TrialDefinition.load(load_name)
print(f"Loaded: {loaded.name}")
print(json.dumps(loaded.to_dict(), indent=2))

## 7 — Auto-Generate Trials from a Scenario Range
Iterate every scenario in the range `SCENARIO_RANGE_START` → `SCENARIO_RANGE_END` (e.g. `17A` to `21D`), group them by map so each map is loaded only once, and for each scenario:

1. Sample two random map waypoints separated by at least `MIN_WAYPOINT_DISTANCE_M` metres (the start and the end).
2. Build a `TrialDefinition` reusing the scenario's weather, vehicle / pedestrian counts (which already encode high vs. low traffic via the A/B/C/D letter), pedestrian behaviour, and seed.
3. Save the trial under `PATH_TO_TRIALS/auto_<scenario_name>/` and drop a `route_preview.png` showing the map waypoints, the picked start (green) and end (red), and the line between them.

Sampling is seeded per-scenario (`SAMPLING_SEED ^ hash(name)`), so re-running the cell is deterministic.

In [ ]:
import random
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

from MIREIA.simulation.bridge import SimulationBridge

# === Range and sampling configuration ===
SCENARIO_RANGE_START = "20C"     # inclusive
SCENARIO_RANGE_END   = "20C"     # inclusive
MIN_WAYPOINT_DISTANCE_M = 30.0   # minimum distance between start and end
SAMPLING_SEED        = 12345      # base seed for waypoint sampling (mixed with scenario name)
TRIAL_NAME_PREFIX    = "auto_"   # prefix for generated trial folders
PREVIEW_IMAGE_NAME   = "route_preview.png"
DISPLAY_PREVIEWS_INLINE = True

# === Discover scenarios in range ===
_RANGE_RE = re.compile(r"^(\d{2})([A-D])_")

def _parse_scenario_id(folder_name: str) -> tuple[int, str] | None:
    m = _RANGE_RE.match(folder_name)
    return (int(m.group(1)), m.group(2)) if m else None

def _id_in_range(sid: tuple[int, str], lo: str, hi: str) -> bool:
    lo_t = (int(lo[:2]), lo[2])
    hi_t = (int(hi[:2]), hi[2])
    return lo_t <= sid <= hi_t

def _weather_label(weather) -> str:
    """Return a short printable label regardless of whether weather is a str or dict."""
    if isinstance(weather, str):
        return weather
    if isinstance(weather, dict):
        # Best-effort: derive a short label from sun altitude + fog density.
        alt = weather.get("sun_altitude_angle", 0)
        fog = weather.get("fog_density", 0)
        tag = "Night" if alt < 0 else "Day"
        if fog > 0:
            tag += f"+Fog{fog:.0f}"
        return tag
    return str(weather)

scenarios_root = Path(Config.PATH_TO_SCENARIOS)
in_range_scenarios = []
for scen_path in sorted(scenarios_root.glob("*/scenario.json")):
    sid = _parse_scenario_id(scen_path.parent.name)
    if sid is None or not _id_in_range(sid, SCENARIO_RANGE_START, SCENARIO_RANGE_END):
        continue
    data = json.loads(scen_path.read_text(encoding="utf-8"))
    data["_folder_name"] = scen_path.parent.name
    in_range_scenarios.append(data)

if not in_range_scenarios:
    raise RuntimeError(f"No scenarios found in range {SCENARIO_RANGE_START}-{SCENARIO_RANGE_END}.")

print(f"Selected {len(in_range_scenarios)} scenarios in range {SCENARIO_RANGE_START}-{SCENARIO_RANGE_END}:")
for s in in_range_scenarios:
    print(
        f"  - {s['_folder_name']:50s} map={s['map_name']:10s} "
        f"weather={_weather_label(s['weather']):24s} nv={s['n_vehicles']:3d} np={s['n_pedestrians']:3d}"
    )

# === Group by map so each map is loaded only once ===
by_map: dict[str, list[dict]] = defaultdict(list)
for s in in_range_scenarios:
    by_map[s["map_name"]].append(s)

print(f"\nMaps to load: {sorted(by_map.keys())}")

# === Helpers ===
def _sample_start_end(wps_xy: np.ndarray, min_dist: float, rng: random.Random) -> tuple[int, int] | None:
    """Random rejection-sampling of (start, end) waypoint indices separated by at least `min_dist`."""
    n = len(wps_xy)
    if n < 2:
        return None
    for _ in range(2000):
        i = rng.randrange(n)
        j = rng.randrange(n)
        if i == j:
            continue
        if np.linalg.norm(wps_xy[i] - wps_xy[j]) >= min_dist:
            return i, j
    return None

def _save_route_preview(trial_obj, wps_xy: np.ndarray, start_xy: np.ndarray, end_xy: np.ndarray,
                        scen: dict, out_path: Path):
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.scatter(wps_xy[:, 0], wps_xy[:, 1], s=1, c="#bbbbbb", zorder=1)
    ax.plot([start_xy[0], end_xy[0]], [start_xy[1], end_xy[1]],
            color="#00BFFF", linewidth=1.5, alpha=0.9, zorder=2)
    ax.scatter([start_xy[0]], [start_xy[1]], s=140, c="#00CC00",
               edgecolors="black", linewidths=0.7, zorder=3, label="start")
    ax.scatter([end_xy[0]],   [end_xy[1]],   s=140, c="#CC0000",
               edgecolors="black", linewidths=0.7, zorder=3, label="end")
    ax.legend(loc="upper right")
    ax.set_aspect("equal")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_title(
        f"{trial_obj.name}\n"
        f"map={scen['map_name']}  weather={_weather_label(scen.get('weather','?'))}  "
        f"nv={scen.get('n_vehicles','?')}  np={scen.get('n_pedestrians','?')}"
    )
    plt.tight_layout()
    fig.savefig(out_path, dpi=120)
    return fig

# === Iterate by map ===
created = []
for map_name, scenarios in by_map.items():
    current_map = client.get_world().get_map().name.split("/")[-1]
    if current_map != map_name:
        print(f"\nLoading map {map_name} (current: {current_map})...")
        client.load_world(map_name)
    map_world = client.get_world()
    print(f"\n=== Map {map_name}: generating {len(scenarios)} trial(s) ===")

    bridge = SimulationBridge(map_world)
    waypoint_collection = bridge.get_waypoints()
    wps = waypoint_collection.waypoints
    if not wps:
        print(f"[skip] No waypoints available for map {map_name}.")
        continue
    wps_xy = np.array([(wp.x, wp.y) for wp in wps], dtype=np.float64)

    for scen in scenarios:
        scen_name = scen["_folder_name"]
        # Per-scenario deterministic seed so re-runs are stable.
        rng = random.Random(SAMPLING_SEED ^ (hash(scen_name) & 0xFFFFFFFF))

        chosen = _sample_start_end(wps_xy, MIN_WAYPOINT_DISTANCE_M, rng)
        if chosen is None:
            print(f"  [skip] {scen_name}: could not sample two separated waypoints "
                  f"(min distance {MIN_WAYPOINT_DISTANCE_M:.1f}m may be too strict)")
            continue
        start_idx, end_idx = chosen
        start_wp = wps[start_idx]
        end_wp   = wps[end_idx]

        # WaypointState is map-space (y-up). CARLA Location is y-down -> negate y.
        route_start = (float(start_wp.x), float(-start_wp.y), 0.0)
        route_end   = (float(end_wp.x),   float(-end_wp.y),   0.0)

        trial = TrialDefinition(
            name=f"{TRIAL_NAME_PREFIX}{scen_name}",
            route_start=route_start,
            route_end=route_end,
            description=f"Auto-generated trial from scenario {scen_name}.",
            map_name=map_name,
            weather=scen.get("weather", "ClearNoon"),
            n_vehicles=int(scen.get("n_vehicles", 30)),
            n_pedestrians=int(scen.get("n_pedestrians", 20)),
            pct_running=float(scen.get("pct_running", 0.0)),
            pct_crossing=float(scen.get("pct_crossing", 0.0)),
            safe_vehicles=bool(scen.get("safe_vehicles", True)),
            seed=int(scen.get("seed", 42)),
            sync_mode=True,
            fixed_delta=Config.SIM_FIXED_DELTA_SECONDS,
        )
        trial.save()

        preview_path = Path(trial.folder_path) / PREVIEW_IMAGE_NAME
        fig = _save_route_preview(trial, wps_xy, wps_xy[start_idx], wps_xy[end_idx], scen, preview_path)
        if DISPLAY_PREVIEWS_INLINE:
            plt.show()
        else:
            plt.close(fig)

        print(f"  + {trial.name:50s} -> {preview_path}")
        created.append(trial.name)

print(f"\nDone. Generated {len(created)} trial(s).")